# ETL Silver - Agregados diarios por sub-cuenca de MERGE y SAMeT

Convierte la grilla de Bronze en un agregado por `(fecha, subcuenca, fuente)`: media areal de los
puntos de grilla que caen dentro del poligono de cada sub-cuenca segun `weather.silver.grid_subcuenca`
(punto -> sub-cuenca, generado en local con geopandas, Decision 033). R8 (Decision 019): sin umbral
de exclusion, la cobertura real viaja como columna (`cobertura_pct` = puntos con dato / puntos de la
sub-cuenca). `es_preliminar` marca las filas que todavia se construyeron con la version preliminar
del archivo de origen (MERGE: antes de la regeneracion del mes siguiente; SAMeT: antes de la
regeneracion con ERA5 a los ~7 dias); se recalcula cada corrida sobre la ventana incremental.
Gold consume solo `alta_frontera` (Decision 018).

In [ ]:
from datetime import date, timedelta

from delta.tables import DeltaTable
from pyspark.sql import functions as F

MERGE_BRONZE = 'weather.bronze.merge_precip_grid'
SAMET_BRONZE = 'weather.bronze.samet_temp_grid'
GRID_TABLE = 'weather.silver.grid_subcuenca'
PRECIP_TARGET = 'weather.silver.precip_grid_daily'
TEMP_TARGET = 'weather.silver.temp_grid_daily'
QUALITY_TABLE = 'weather.silver.attribute_quality'
THRESHOLD_PCT = 0.90
QUALITY_WINDOW_DAYS = 30

try:
    dbutils.widgets.dropdown('load_mode', 'incremental', ['full', 'incremental'])
    dbutils.widgets.text('incremental_lookback_days', '60')
    load_mode = dbutils.widgets.get('load_mode')
    incremental_lookback_days = int(dbutils.widgets.get('incremental_lookback_days'))
except Exception:
    load_mode, incremental_lookback_days = 'incremental', 60

print(f'load_mode={load_mode}, incremental_lookback_days={incremental_lookback_days}')

In [ ]:
def grid_for(grilla):
    return (
        spark.table(GRID_TABLE)
        .filter(F.col('grilla') == F.lit(grilla))
        .select(F.round('latitude', 3).alias('latitude'), F.round('longitude', 3).alias('longitude'), 'subcuenca')
    )


def expected_points(grid_df):
    return grid_df.groupBy('subcuenca').agg(F.count('*').cast('bigint').alias('puntos_esperados'))


def apply_incremental_window(df, target_table):
    if load_mode == 'full':
        return df
    max_fecha = spark.table(target_table).agg(F.max('fecha').alias('m')).first()['m']
    if max_fecha is None:
        return df
    start = max_fecha - timedelta(days=incremental_lookback_days)
    print(f'{target_table}: ventana incremental desde {start}')
    return df.filter(F.col('fecha') >= F.lit(start))


def merge_target(df, target_table):
    if df.limit(1).count() == 0:
        print(f'{target_table}: nada que mergear')
        return
    DeltaTable.forName(spark, target_table).alias('t').merge(
        df.alias('s'),
        't.fecha = s.fecha AND t.subcuenca = s.subcuenca AND t.fuente = s.fuente',
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


def build_quality(df, source_table, source_name, attribute_name, notes):
    end_date = date.today() - timedelta(days=1)
    start_date = end_date - timedelta(days=QUALITY_WINDOW_DAYS - 1)
    win = df.filter((F.col('fecha') >= F.lit(start_date)) & (F.col('fecha') <= F.lit(end_date)) & (F.col('subcuenca') == F.lit('alta_frontera')))
    return (
        win.agg(
            F.min('fecha').alias('evaluation_start_date'),
            F.max('fecha').alias('evaluation_end_date'),
            F.countDistinct(F.when(F.col(attribute_name).isNotNull(), F.col('fecha'))).cast('bigint').alias('observed_days'),
        )
        .withColumn('expected_days', F.when(F.col('evaluation_start_date').isNull(), F.lit(0)).otherwise(F.datediff(F.col('evaluation_end_date'), F.col('evaluation_start_date')) + F.lit(1)).cast('bigint'))
        .withColumn('missing_days', F.greatest(F.col('expected_days') - F.col('observed_days'), F.lit(0)).cast('bigint'))
        .withColumn('missing_pct', F.when(F.col('expected_days') == 0, F.lit(1.0)).otherwise(F.col('missing_days') / F.col('expected_days')))
        .withColumn('threshold_pct', F.lit(THRESHOLD_PCT))
        .withColumn('is_usable', F.col('missing_pct') <= F.col('threshold_pct'))
        .withColumn('source_layer', F.lit('silver'))
        .withColumn('source_table', F.lit(source_table))
        .withColumn('source_name', F.lit(source_name))
        .withColumn('attribute_name', F.lit(attribute_name))
        .withColumn('grain', F.lit('global_source_daily'))
        .withColumn('evaluated_at', F.current_timestamp())
        .withColumn('notes', F.lit(notes))
        .withColumn('created_at', F.current_timestamp())
        .withColumn('updated_at', F.current_timestamp())
        .select('source_layer', 'source_table', 'source_name', 'attribute_name', 'grain', 'evaluation_start_date', 'evaluation_end_date', 'expected_days', 'observed_days', 'missing_days', 'missing_pct', 'threshold_pct', 'is_usable', 'evaluated_at', 'notes', 'created_at', 'updated_at')
    )


def merge_quality(quality_df):
    DeltaTable.forName(spark, QUALITY_TABLE).alias('t').merge(
        quality_df.alias('s'),
        't.source_table = s.source_table AND t.attribute_name = s.attribute_name AND t.grain = s.grain',
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [ ]:
# --- MERGE (precipitacion) ---------------------------------------------------------------
# Regla es_preliminar: el archivo diario de D se publica ~02:40 UTC de D+1 y CPTEC regenera el mes
# completo en los primeros dias del mes siguiente (observado: dias 1-6, a horas variables). Un
# umbral fijo por mes clasifica mal los meses regenerados el dia 1 (julio 2026 se regenero el
# 2026-08-01 12:43 y quedaba 'preliminar' hasta el dia 7). Regla robusta: es preliminar si el
# archivo NO fue tocado despues de su publicacion inicial, o sea source_last_modified < D+2.
grid_merge = grid_for('merge_0p1')
expected_merge = expected_points(grid_merge)

bronze_merge = (
    spark.table(MERGE_BRONZE)
    .withColumn('latitude', F.round('latitude', 3))
    .withColumn('longitude', F.round('longitude', 3))
)
bronze_merge = apply_incremental_window(bronze_merge, PRECIP_TARGET)

precip_daily = (
    bronze_merge.alias('b')
    .join(grid_merge.alias('g'), ['latitude', 'longitude'], 'inner')
    .groupBy('fecha', 'subcuenca')
    .agg(
        F.avg('prec_mm').alias('prec_media_mm'),
        F.max('prec_mm').alias('prec_max_mm'),
        F.count('prec_mm').cast('bigint').alias('puntos_grilla'),
        F.sum(F.when(F.col('nest') > 0, 1).otherwise(0)).cast('bigint').alias('puntos_con_pluviometro'),
        F.sum('nest').cast('bigint').alias('pluviometros'),
        F.max('source_last_modified').alias('source_last_modified'),
    )
    .join(expected_merge, 'subcuenca', 'left')
    .withColumn('fuente', F.lit('merge'))
    .withColumn('cobertura_pct', F.when(F.col('puntos_esperados') > 0, F.col('puntos_grilla') / F.col('puntos_esperados')))
    .withColumn('es_preliminar', F.col('source_last_modified') < F.to_timestamp(F.date_add(F.col('fecha'), 2)))
    .withColumn('source_table', F.lit(MERGE_BRONZE))
    .withColumn('processed_at', F.current_timestamp())
    .withColumn('updated_at', F.current_timestamp())
    .select('fecha', 'subcuenca', 'fuente', 'prec_media_mm', 'prec_max_mm', 'puntos_grilla', 'puntos_esperados', 'cobertura_pct',
            'puntos_con_pluviometro', 'pluviometros', 'source_last_modified', 'es_preliminar', 'source_table', 'processed_at', 'updated_at')
)
merge_target(precip_daily, PRECIP_TARGET)
merge_quality(build_quality(spark.table(PRECIP_TARGET), PRECIP_TARGET, 'precip_grid_daily', 'prec_media_mm',
                            f'MERGE (CPTEC) alta_frontera; ventana de calidad de {QUALITY_WINDOW_DAYS} dias; metrica informativa (R8), no bloquea publicacion'))
spark.table(PRECIP_TARGET).groupBy('subcuenca').agg(F.min('fecha').alias('inicio'), F.max('fecha').alias('fin'), F.count('*').alias('rows'), F.avg('cobertura_pct').alias('cobertura_media')).orderBy('subcuenca').show(truncate=False)

In [ ]:
# --- SAMeT (temperatura) -----------------------------------------------------------------
# temp_media/max/min_c = media areal de tmed/tmax/tmin (la magnitud comparable con el agregado
# por estacion es la media; los extremos absolutos de la grilla van aparte en *_abs_c).
# Regla es_preliminar: SAMeT se regenera con ERA5 ~7 dias despues de la fecha (READ-ME oficial;
# observado a las 03:00 UTC de D+7). Un archivo modificado antes de D+7 es la version preliminar
# (observaciones + pronostico numerico).
grid_samet = grid_for('samet_0p05')
expected_samet = expected_points(grid_samet)

bronze_samet = (
    spark.table(SAMET_BRONZE)
    .withColumn('latitude', F.round('latitude', 3))
    .withColumn('longitude', F.round('longitude', 3))
)
bronze_samet = apply_incremental_window(bronze_samet, TEMP_TARGET)

temp_daily = (
    bronze_samet.alias('b')
    .join(grid_samet.alias('g'), ['latitude', 'longitude'], 'inner')
    .groupBy('fecha', 'subcuenca')
    .agg(
        F.avg('tmed_c').alias('temp_media_c'),
        F.avg('tmax_c').alias('temp_max_c'),
        F.avg('tmin_c').alias('temp_min_c'),
        F.max('tmax_c').alias('temp_max_abs_c'),
        F.min('tmin_c').alias('temp_min_abs_c'),
        F.count('tmed_c').cast('bigint').alias('puntos_grilla'),
        (F.sum('nobs_tmed') + F.sum('nobs_tmax') + F.sum('nobs_tmin')).cast('bigint').alias('nobs_total'),
        F.max('source_last_modified').alias('source_last_modified'),
    )
    .join(expected_samet, 'subcuenca', 'left')
    .withColumn('fuente', F.lit('samet'))
    .withColumn('cobertura_pct', F.when(F.col('puntos_esperados') > 0, F.col('puntos_grilla') / F.col('puntos_esperados')))
    .withColumn('es_preliminar', F.col('source_last_modified') < F.to_timestamp(F.date_add(F.col('fecha'), 7)))
    .withColumn('source_table', F.lit(SAMET_BRONZE))
    .withColumn('processed_at', F.current_timestamp())
    .withColumn('updated_at', F.current_timestamp())
    .select('fecha', 'subcuenca', 'fuente', 'temp_media_c', 'temp_max_c', 'temp_min_c', 'temp_max_abs_c', 'temp_min_abs_c',
            'puntos_grilla', 'puntos_esperados', 'cobertura_pct', 'nobs_total', 'source_last_modified', 'es_preliminar', 'source_table', 'processed_at', 'updated_at')
)
merge_target(temp_daily, TEMP_TARGET)
merge_quality(build_quality(spark.table(TEMP_TARGET), TEMP_TARGET, 'temp_grid_daily', 'temp_media_c',
                            f'SAMeT (CPTEC) alta_frontera; ventana de calidad de {QUALITY_WINDOW_DAYS} dias; metrica informativa (R8), no bloquea publicacion'))
spark.table(TEMP_TARGET).groupBy('subcuenca').agg(F.min('fecha').alias('inicio'), F.max('fecha').alias('fin'), F.count('*').alias('rows'), F.avg('cobertura_pct').alias('cobertura_media')).orderBy('subcuenca').show(truncate=False)